# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [15]:
X = spaceship.drop(
    columns=["PassengerId", "Name", "Cabin", "Transported"]
)
y = spaceship["Transported"]

In [16]:
num_features = [
    "Age", "RoomService", "FoodCourt",
    "ShoppingMall", "Spa", "VRDeck"
]

cat_features = [
    "HomePlanet", "CryoSleep",
    "Destination", "VIP"
]

In [17]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features)
    ]
)

**Perform Train Test Split**

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [20]:
bagging_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", BaggingClassifier(
        estimator=LogisticRegression(max_iter=1000),
        n_estimators=100,
        random_state=42
    ))
])

bagging_model.fit(X_train, y_train)
y_pred_bag = bagging_model.predict(X_test)

bagging_acc = accuracy_score(y_test, y_pred_bag)
bagging_acc

0.7872340425531915

- Random Forests

In [21]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_acc


0.7843588269120184

- Gradient Boosting

In [22]:
gb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

gb_acc = accuracy_score(y_test, y_pred_gb)
gb_acc


0.8027602070155262

- Adaptive Boosting

In [23]:
ada_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", AdaBoostClassifier(
        n_estimators=200,
        learning_rate=0.5,
        random_state=42
    ))
])

ada_model.fit(X_train, y_train)
y_pred_ada = ada_model.predict(X_test)

ada_acc = accuracy_score(y_test, y_pred_ada)
ada_acc


0.7607820586543991

In [24]:
results = pd.DataFrame({
    "Model": [
        "Bagging",
        "Random Forest",
        "Gradient Boosting",
        "AdaBoost"
    ],
    "Accuracy": [
        bagging_acc,
        rf_acc,
        gb_acc,
        ada_acc
    ]
})

results.sort_values(by="Accuracy", ascending=False)


,Model,Accuracy
2,Gradient Boosting,0.802760
0,Bagging,0.787234
1,Random Forest,0.784359
3,AdaBoost,0.760782


Which model is the best and why?

Gradient Boosting handles non-linear interactions well
Reduces bias focuses on hard-to-classify passengers, improving generalization.
Therefore is robust to mixed feature types
Lower overfitting risk than Random Forest